## Unity Catalog ABAC — attribute-based access control

Per-table row filters and column masks work — but they don't scale to *hundreds* of tables. **Unity Catalog ABAC** lets you define a policy **once** and apply it everywhere a table or column carries a specific **tag**.

**Three building blocks:**

- **Tags** — labels on catalogs / schemas / tables / columns (`pii_class = high`, `data_domain = cards`, …).
- **Policies** — central rules like "mask any column tagged `pii_class = high` for everyone outside `compliance`."
- **Evaluation** — at query time UC reads the object's tags, finds matching policies, and applies them **on top of** the principal's GRANTs.

```sql
ALTER TABLE fintech_dev.silver.customers ALTER COLUMN email
  SET TAGS ('pii_class' = 'high');

CREATE POLICY fintech_dev.security.mask_high_pii
  ON COLUMN MATCH TAGS ('pii_class' = 'high')
  USING fintech_dev.security.mask_email(value)
  WHEN NOT is_account_group_member('compliance');
```

Add a new PII column on a new table, tag it `pii_class = high`, and the **existing policy already covers it** — no per-table mask to remember.

**The exam tell:** *"centralised row-level filtering and column masking for sensitive data across many objects"* → **Unity Catalog ABAC**. Per-table masks/filters are the *manual* answer; ABAC + tags is the *scale* answer. The distinguishing word is usually **"across hundreds of tables"** or **"centralised."**
